# JEPA self-supervised pretraining for PVT v2

Implements the design in **`docs/JEPA_GUIDE.md`** (read it first — it is the
step-by-step recipe this notebook follows):

- **Masking**: 32px mask units aligned to the stage-4 7x7 grid (Hiera-style);
  applied as a learnable token after the stage-1 patch embed (SimMIM-style).
  No token dropping — PVT's conv stems / SRA / DWConv need a contiguous grid.
- **Context encoder**: PVT v2, **MoE off** (pretrain dense, upcycle experts at
  fine-tune). **Target encoder**: EMA copy, full image, no grad.
- **Predictor**: narrow ViT (384d x 6) on the 7x7 grid; smooth-L1 on LayerNorm'd
  target features at masked positions (I-JEPA).
- **Collapse watch**: `target_std` must stay well above 0 (healthy: O(0.5-2)).

No MoE backend needed — this notebook runs on any CUDA box with torch >= 2.4.

In [ ]:
# Environment check — plain Jupyter on B200 (sm_100) or RTX 5090 (sm_120).
# Credentials: export HF_TOKEN / WANDB_API_KEY in the shell that starts Jupyter.
import importlib
import subprocess
import sys

import torch

print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})")
else:
    print("WARNING: no GPU visible — training will not be practical.")

_torch_minor = tuple(int(p) for p in torch.__version__.split("+")[0].split(".")[:2])
if _torch_minor < (2, 5):
    print("NOTE: torch >= 2.5 recommended (fast SDPA GQA path; >=2.4 for fused RMSNorm). "
          "The code falls back gracefully but slower.")

_required = ["pytorch_lightning", "torchmetrics", "timm", "datasets", "transformers",
             "huggingface_hub", "pandas", "matplotlib", "fvcore", "wandb"]
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
if _missing:
    print(f"Installing: {_missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

In [ ]:
# Make the repo importable (notebooks/ lives one level under the repo root).
import pathlib
import sys

cwd = pathlib.Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert (REPO_ROOT / "pvt_moe").is_dir(), f"pvt_moe package not found under {REPO_ROOT}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pvt_moe import default_config, merge_config, validate_config
from pvt_moe.data import build_dataloaders
from pvt_moe.engine import LitClassifier, build_ssl_trainer, build_trainer, setup_environment
from pvt_moe.models import build_model
from pvt_moe.utils import (
    count_flops,
    count_params,
    expert_utilization,
    plot_expert_utilization,
    plot_training_curves,
)

print(f"pvt_moe loaded from {REPO_ROOT}")

from pvt_moe.ssl import LitJEPA, LitProbe, sample_batch_masks, upsample_mask

In [ ]:
# ============================== CONFIG =====================================
cfg = merge_config(default_config(), {
    "mode": "scratch",
    "use_wandb": True,
    "wandb_project": "pvt-jepa-imagenet",
    "batch_size": 512,               # raise with grad accumulation if desired
    "num_workers": 12,
    "run_name": "jepa_v1_in1k",

    "model": {
        "pretrained_hf_id": None,     # SSL trains from scratch
        "ablation": {
            "use_moe": False,         # ALWAYS dense for SSL (see guide)
            "moe_placement": [[], [], [], []],
            "use_rope": True,         # positional signal for stage 4
            "rope_placement": [[], [], [], [0, 1]],
            "rope_theta": None,   # None = per-mode default (mixed 10 / axial 50)
        },
    },
    "dataset": {"name": "imagenet-1k"},

    "ssl": {
        "epochs": 100,               # 300+ for a serious run
        "lr": 1.5e-3,                # tuned for global batch ~2048; scale linearly
        "final_lr": 1e-6,
        "warmup_epochs": 15,
        "weight_decay": 0.04, "weight_decay_end": 0.4,
        "ema_momentum": 0.996, "ema_momentum_end": 1.0,
        "mask_n_blocks": 4, "mask_block_area": [0.10, 0.20],
        "mask_aspect_ratio": [0.75, 1.5],
        "predictor_dim": 384, "predictor_depth": 6, "predictor_heads": 6,
        "grad_clip": 3.0,
    },
})

cfg = validate_config(cfg)
print("run:", cfg["run_name"])

In [ ]:
device = setup_environment(cfg)

In [ ]:
# SSL data: crop + flip only (build_dataloaders swaps the train transform).
train_loader, val_loader = build_dataloaders(cfg, ssl=True)
xb, _ = next(iter(train_loader))
print("batch:", xb.shape)

In [ ]:
# Visualize the masking: 32px units on the 7x7 grid.
import matplotlib.pyplot as plt
import torch

masks = sample_batch_masks(8, grid=7, generator=torch.Generator().manual_seed(0))
fig, axes = plt.subplots(1, 8, figsize=(16, 2.2))
for ax, m in zip(axes, masks):
    ax.imshow(m.view(7, 7), cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(f"{m.float().mean():.0%}", fontsize=9)
    ax.axis("off")
plt.suptitle("sampled masks (dark = masked / predicted)")
plt.show()

In [ ]:
# Sanity: full JEPA math on one small batch before committing GPU-days.
import torch
import torch.nn.functional as F

jepa = LitJEPA(cfg)
_x = xb[:4].to(device)
jepa = jepa.to(device)

_mask = sample_batch_masks(4, grid=jepa.mask_grid).to(device)
_s1 = upsample_mask(_mask, jepa.mask_grid, jepa.stage1_grid)
_ctx, _aux = jepa.context.forward_features(_x, return_tokens=True,
                                           stage1_token_mask=_s1,
                                           mask_token=jepa.input_mask_token)
assert _aux is None, "SSL backbone must be dense"
with torch.no_grad():
    _tgt, _ = jepa.target.forward_features(_x, return_tokens=True)
_pred = jepa.predictor(_ctx, _mask)
_loss = F.smooth_l1_loss(_pred[_mask], F.layer_norm(_tgt, (_tgt.shape[-1],))[_mask])
assert torch.isfinite(_loss)
print(f"JEPA sanity OK — initial loss {_loss.item():.4f}")

In [ ]:
trainer = build_ssl_trainer(cfg)
trainer.fit(jepa, train_loader)

In [ ]:
import os

backbone_path = os.path.join(cfg["checkpoint_root"], cfg["run_name"], "jepa_backbone.pt")
jepa.save_backbone(backbone_path)

In [ ]:
# Linear probe: frozen backbone + one linear layer (the standard SSL metric).
import pytorch_lightning as pl

probe_epochs = 20
probe = LitProbe(jepa.context, num_classes=cfg["dataset"]["num_classes"],
                 lr=1e-3, epochs=probe_epochs)
probe_train, probe_val = build_dataloaders(cfg)   # standard supervised transforms
probe_trainer = pl.Trainer(max_epochs=probe_epochs, accelerator="auto", devices=1,
                           precision=cfg["precision"], logger=False,
                           enable_checkpointing=False)
probe_trainer.fit(probe, probe_train, probe_val)

## Handoff to supervised fine-tuning

In `01_train_supervised.ipynb`, point the config at the saved backbone:

```python
cfg = merge_config(default_config(), {
    "mode": "ssl_init",
    "ckpt_path": "<checkpoint_root>/jepa_v1_in1k/jepa_backbone.pt",
    "model": {"pretrained_hf_id": None},   # weights come from JEPA, not HF
    # MoE upcycling from an SSL backbone: experts start from random init
    # (no dense FFN teacher for the MoE blocks) — expect a slower first
    # few epochs than HF-seeded runs.
})
```

Reference results to compare against (I-JEPA, ViT-B/16, 600 ep): ~72% linear probe.
A 100-epoch PVT-B1 run will land far lower — track the *trend*, not the absolute.